In [2]:
# ==========================================================
# CNN FLOWERS CLASSIFICATION – END TO END (FIXED & SAFE)
# ==========================================================

import os
import tarfile
import urllib.request
import pathlib
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    RandomFlip, RandomRotation, RandomContrast
)
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------
# CONFIG
import os, tarfile, urllib.request, pathlib
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# ---------------- CONFIG ----------------
DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_DIR = "flower_data"
IMAGE_SIZE = (180, 180)
BATCH_SIZE = 32
EPOCHS = 30

CLASSES = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]
CLASS_TO_INDEX = {c: i for i, c in enumerate(CLASSES)}

# ---------------- DOWNLOAD ----------------
if not os.path.exists(DATA_DIR):
    tgz, _ = urllib.request.urlretrieve(DATA_URL, "flowers.tgz")
    with tarfile.open(tgz, "r:gz") as tar:
        tar.extractall(DATA_DIR)

FLOWER_DIR = pathlib.Path(DATA_DIR) / "flower_photos"

# ---------------- LOAD DATA ----------------
X, y = [], []

for cls in CLASSES:
    for img_path in (FLOWER_DIR / cls).glob("*.jpg"):
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMAGE_SIZE)
        img = tf.keras.preprocessing.image.img_to_array(img)
        X.append(img)
        y.append(CLASS_TO_INDEX[cls])

X = np.array(X, dtype="float32") / 255.0
y = np.array(y, dtype="int32")

# ---------------- SPLIT ----------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------- AUGMENTATION ----------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.2),
])

# ---------------- BEST CNN MODEL ----------------
model = models.Sequential([
    data_augmentation,

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(CLASSES), activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ---------------- TRAIN ----------------
model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1
)

# ---------------- EVALUATE ----------------
loss, acc = model.evaluate(X_test, y_test)
print("Final Test Accuracy:", acc)

# ---------------- SAVE ----------------
model.save("flower_cnn_model.h5")
np.save("classes.npy", np.array(CLASSES))

print("Model saved successfully")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_1 (Sequential)       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 14s 105ms/step - accuracy: 0.3935 - loss: 4.1788 - val_accuracy: 0.2245 - val_loss: 9.3904
Epoch 2/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.4801 - loss: 1.2591 - val_accuracy: 0.2245 - val_loss: 12.5775
Epoch 3/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 91ms/step - accuracy: 0.5358 - loss: 1.1824 - val_accuracy: 0.2245 - val_loss: 6.7377
Epoch 4/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.5300 - loss: 1.1944 - val_accuracy: 0.2925 - val_loss: 3.0433
Epoch 5/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.5686 - loss: 1.1286 - val_accuracy: 0.3469 - val_loss: 1.8278
Epoch 6/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 10s 94ms/step - accuracy: 0.5627 - loss: 1.0857 - val_accuracy: 0.5170 - val_loss: 1.2189
Epoch 7/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 94ms/step - accuracy: 0.5703 - loss: 1.0993 - val_accuracy: 0.5884 - val_loss: 1.1441
Epoch 8/30
83/83 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.6148 - loss: 1.0319 - val_accuracy: 0.6327

Final Test Accuracy: 0.7247956395149231
Model saved successfully


In [3]:
print(CLASSES)

['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
